# Task 1 — Classification (§2.1 BatchNorm · §2.2 Dropout · §2.4 Feature Maps)

**Colab setup:** Runtime → Change runtime type → **T4 GPU**

**After this notebook you will have:**
- `classifier.pth` saved directly to your Google Drive
- W&B runs logged for §2.1 and §2.2
- `CLASSIFIER_DRIVE_ID` updated in `models/multitask.py` and pushed to GitHub
- Ready to submit **4.1b / 4.1c** to Gradescope

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CKPT_DIR}")

## 2 · Setup

In [ ]:
import os

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"

GITHUB_TOKEN    = "PASTE_YOUR_GITHUB_TOKEN_HERE"
GITHUB_USERNAME = "usnaveen"
GITHUB_REPO     = "A2_Deep_Learning"

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git da6401_assignment_2
%cd da6401_assignment_2

!pip install -q wandb albumentations scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

## 3 · Download Dataset

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")

## 4 · §2.2 — Dropout Experiment
3 runs (p=0, 0.2, 0.5). Checkpoints saved to Drive after each run.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"

!python train.py --task classify --experiment exp-dropout --dropout 0.0 --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 --checkpoint-dir {CKPT_DIR}
!python train.py --task classify --experiment exp-dropout --dropout 0.2 --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 --checkpoint-dir {CKPT_DIR}
!python train.py --task classify --experiment exp-dropout --dropout 0.5 --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 --checkpoint-dir {CKPT_DIR}

## 5 · §2.1 — BatchNorm Ablation

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
!python train.py --task classify --experiment exp-batchnorm --no-bn --dropout 0.5 --epochs 120 --lr 1e-4 --batch-size 32 --num-workers 2 --checkpoint-dir {CKPT_DIR}

## 6 · §2.4 — Feature Map Visualization

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
!python inference.py --mode featuremaps --num-workers 2 --checkpoint-dir {CKPT_DIR}

## 7 · Verify F1 Score

In [ ]:
import torch
from sklearn.metrics import f1_score
from data.pets_dataset import create_dataloaders
from models.classification import VGG11Classifier

CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, val_loader, _ = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
model.load_state_dict(torch.load(f"{CKPT_DIR}/classifier.pth", map_location=device, weights_only=False))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        logits = model(batch["image"].to(device))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(batch["label"].tolist())

f1 = f1_score(all_labels, all_preds, average="macro")
print(f"\n✅  Val Macro-F1 = {f1:.4f}")
print("   need > 0.50 for 4.1b · > 0.80 for 4.1c")

## 8 · Get Drive File ID + Make Public
This cell automatically gets the `classifier.pth` file ID from Drive and makes it publicly accessible — no manual sharing needed.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive = build('drive', 'v3')

# Find classifier.pth in Drive
results = drive.files().list(
    q="name='classifier.pth' and trashed=false",
    fields="files(id, name)"
).execute()

files = results.get('files', [])
if not files:
    print("❌  classifier.pth not found in Drive — did training finish?")
else:
    CLASSIFIER_DRIVE_ID = files[0]['id']
    # Make it public
    drive.permissions().create(
        fileId=CLASSIFIER_DRIVE_ID,
        body={'type': 'anyone', 'role': 'reader'}
    ).execute()
    print(f"✅  CLASSIFIER_DRIVE_ID = {CLASSIFIER_DRIVE_ID}")
    print(f"   Share URL: https://drive.google.com/file/d/{CLASSIFIER_DRIVE_ID}/view")

## 9 · Update `multitask.py` + Push to GitHub

In [ ]:
import re

multitask_path = "models/multitask.py"
with open(multitask_path) as f:
    content = f.read()

content = re.sub(
    r'CLASSIFIER_DRIVE_ID\s*=\s*"[^"]*"',
    f'CLASSIFIER_DRIVE_ID = "{CLASSIFIER_DRIVE_ID}"',
    content
)

with open(multitask_path, "w") as f:
    f.write(content)

print(f"✅  Updated multitask.py with CLASSIFIER_DRIVE_ID")

!git config user.email "naveen@student.iitm.ac.in"
!git config user.name  "Naveen"
!git add models/multitask.py
!git commit -m "task1: set CLASSIFIER_DRIVE_ID for Gradescope"
!git push

print("\n🎉  Done! Submit to Gradescope to check 4.1b / 4.1c")